# 04 · XGBoost — the countdown

**The only job XGBoost does in this system.** It does not rank ATMs and it does
not detect mules; it estimates **minutes until withdrawal** for the top-ranked
candidate, together with a confidence band.

Gradient boosting is the right tool here for an unglamorous reason: this is a
small-n, mixed-type, tabular regression (~65k rows, 80 heterogeneous features).
A neural network needs far more data to match boosted trees on that shape and
gives up interpretability in exchange.

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

# Works whether the kernel starts in notebooks/ or at the project root.
ROOT = Path.cwd()
while not (ROOT / "engine").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "engine"))

DATA = ROOT / "data"
MODELS = ROOT / "models"

# The published numbers. Everything below is checked against this file.
METRICS = json.loads((DATA / "metrics.json").read_text())

def check(label, got, expected, tol=5e-3):
    """Print whether a freshly computed figure matches the published one."""
    ok = abs(float(got) - float(expected)) <= tol
    print(f"  {'MATCH   ' if ok else 'MISMATCH'}  {label:38s} "
          f"notebook={float(got):.4f}  published={float(expected):.4f}")
    return ok

print("project root :", ROOT)
print("metrics from :", METRICS["_about"][:60], "...")

## 1 · The model and its inputs

In [ ]:
import pickle
with open(MODELS / "xgb_cashout.pkl", "rb") as f:
    bundle = pickle.load(f)

reg    = bundle["regressor"]
reg_lo = bundle["regressor_lo"]
reg_hi = bundle["regressor_hi"]

print("regressor      :", type(reg).__name__)
print("n_estimators   :", reg.n_estimators)
print("max_depth      :", reg.max_depth)
print("learning_rate  :", reg.learning_rate)
print("quantile models:", type(reg_lo).__name__, "(q05) /", type(reg_hi).__name__, "(q95)")

from feature_builder import TABULAR_FEATURE_NAMES, EMBEDDING_DIM
print(f"\nfeature vector: {EMBEDDING_DIM}-dim GNN embedding "
      f"+ {len(TABULAR_FEATURE_NAMES)} tabular = "
      f"{EMBEDDING_DIM + len(TABULAR_FEATURE_NAMES)}-dim")

The quantile pair is the part that matters operationally. A single number hides
how much of the delay is genuinely unknowable; an officer being dispatched needs
"next 20-70 minutes", not "at 14:32".

## 2 · Published performance

In [ ]:
loc = METRICS["location"]
rows = [
    ("Test MAE (minutes)",        loc["time_mae_minutes"]),
    ("Mean-prediction baseline",  loc["time_baseline_mae"]),
    ("R-squared",                 loc["time_r2"]),
    ("q05-q95 coverage",          loc["interval_coverage"]),
    ("Interval width (minutes)",  loc["interval_width_min"]),
    ("Median lead time (minutes)",loc["lead_time_median_min"]),
    ("Actionable rate (>=15 min)",loc["lead_actionable_rate"]),
]
display(pd.DataFrame(rows, columns=["metric", "value"]).round(4))

improve = loc["time_baseline_mae"] - loc["time_mae_minutes"]
print(f"\nBeats the mean-guess baseline by {improve:.2f} minutes "
      f"({100 * improve / loc['time_baseline_mae']:.1f}% lower error).")

## 3 · Error against the ceiling — the one model with real headroom

In [ ]:
CEILING_MAE = 9.38    # conditioned on observables (OVERNIGHT_ML_AUDIT.md, section 8)
CEILING_R2  = 0.445

achieved = loc["time_mae_minutes"]
baseline = loc["time_baseline_mae"]

fig, ax = plt.subplots(figsize=(8, 2.2))
ax.barh(["MAE (min)"], [CEILING_MAE], color="#3d8b00", label="irreducible")
ax.barh(["MAE (min)"], [achieved - CEILING_MAE], left=[CEILING_MAE],
        color="#c77b30", label="remaining headroom")
ax.barh(["MAE (min)"], [baseline - achieved], left=[achieved],
        color="#dfe6e9", label="already beaten vs baseline")
ax.axvline(achieved, color="#2c3e50", lw=1.5)
ax.text(achieved, 0.35, f" achieved {achieved:.2f}", fontsize=8, va="bottom")
ax.set_xlabel("mean absolute error (minutes)"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

print(f"achieved MAE {achieved:.2f}   ceiling {CEILING_MAE:.2f}   "
      f"headroom {achieved - CEILING_MAE:.2f} min")
print(f"achieved R2  {loc['time_r2']:.3f}   ceiling {CEILING_R2:.3f}")

Unlike the other two components, this one is **not** at its ceiling — and the
project says so. The gap is a two-regime mixture: a crew either has a runner
already at the machine or has to travel, and which regime applies is a draw the
model cannot observe even knowing the crew's speed exactly.

Adding observed intra-chain hop timing (median gap, fastest gap, elapsed span —
all visible the moment a complaint is traced) moved MAE 12.04 -> 11.82 and
R-squared 0.155 -> 0.167. Real, but small.

## 4 · Why a band and not a point estimate

In [ ]:
cov, width = loc["interval_coverage"], loc["interval_width_min"]

fig, ax = plt.subplots(figsize=(8, 2))
ax.barh(["q05-q95"], [cov], color="#3d8b00")
ax.axvline(0.90, color="#c0392b", ls="--", label="nominal 90%")
ax.set_xlim(0, 1); ax.set_xlabel("empirical coverage"); ax.legend(fontsize=8)
ax.set_title(f"Interval covers {cov:.1%} of true delays, {width:.1f} min wide")
plt.tight_layout(); plt.show()

print(f"Nominal coverage 90%, empirical {cov:.1%} - the band is somewhat")
print("narrower than advertised, which is the honest reading and is reported as such.")

## 5 · What this buys operationally

In [ ]:
lead   = loc["lead_time_median_min"]
action = loc["lead_actionable_rate"]

fig, ax = plt.subplots(figsize=(7.5, 2))
ax.barh(["cases"], [action], color="#3d8b00", label=f"actionable ({action:.1%})")
ax.barh(["cases"], [1 - action], left=[action], color="#dfe6e9",
        label="too little lead time")
ax.set_xlim(0, 1); ax.legend(fontsize=8, loc="center right")
ax.set_title(f"Median {lead:.0f} minutes of warning before the predicted withdrawal")
plt.tight_layout(); plt.show()

print(f"median lead time      : {lead:.1f} min")
print(f"actionable (>=15 min) : {action:.1%}")
print(f"inference (mean/max)  : {loc['inference_mean_ms']:.2f} / "
      f"{loc['inference_max_ms']:.2f} ms")

## Summary across every component

Read from `data/metrics.json` rather than typed. A summary table of hand-copied
figures is the exact drift these notebooks exist to prevent -- it was one, and
the retrain that moved every number is what showed why that was a bad idea.

In [ ]:
det, loc, rank = METRICS["detection"], METRICS["location"], METRICS["ranking"]
base, hs = METRICS["detection_baselines"], METRICS["hotspot"]

# ranking["curve"] is a list of rows, one per K.
by_k = {int(r["K"]): r for r in rank["curve"]}

rows = [
    ("GraphSAGE detection",      f"F1 {det['test_f1']:.4f}",
     f"{base['best_non_graph']:.4f} ({base['best_non_graph_model']})"),
    ("Conditional logit, Top-1", f"{by_k[1]['containment']:.4f}",
     f"{by_k[1]['baseline']:.4f} (nearest ATM)"),
    ("Conditional logit, Top-5", f"{rank['top5']:.4f}",
     f"{rank['top5_baseline_distance']:.4f} (nearest 5)"),
    ("Search zone",              f"{loc['zone_containment']:.3f}",
     f"{loc['zone_containment_baseline_nearest3']:.3f} (nearest 3)"),
    ("XGBoost countdown",        f"{loc['time_mae_minutes']:.2f} min MAE",
     f"{loc['time_baseline_mae']:.2f} min (predict the mean)"),
    ("Forward forecast, hit@5",  f"{hs['hit_rate_at_k']['5']:.4f}",
     f"{hs['baseline_historical_density']['hit_rate_at_5']:.4f} (historical density)"),
    ("Forward forecast, PAI@5",  f"{hs['pai_at_k']['5']:.2f}",
     f"{hs['baseline_historical_density']['pai_at_5']:.2f} (historical density)"),
]
summary = pd.DataFrame(rows, columns=["component", "achieved", "baseline it must beat"])
display(summary.set_index("component"))

beat = [r for r in rows if r[0] != "Conditional logit, Top-1"]
print(f"{len(rows)} components, each printed next to the baseline it has to beat.")
print(f"Top-1 ATM ({by_k[1]['containment']:.4f} against {by_k[1]['baseline']:.4f}) is the one")
print("that does not clear its baseline, and it is in the table for that reason.")
print(f"R2 on the countdown is {loc['time_r2']:.4f}: the cashout delay is only")
print("partly knowable from what a complaint can observe, which is why the")
print("forecast ships an interval rather than a point.")